# Ironclad CanSat: descent simulation
Team Ironclad, SEVA FIRST Innovation Challenge 2026 (NAT-004).

Drops the CanSat from 500 m, waits for the firmware to detect the fall, fires the burn wire,
opens the parachute and steps the fall forward every 0.01 s until it lands.

**All results are simulated, using estimated inputs.** Change the numbers in the first cell after weighing the real build.
This notebook uses fixed air density and no wind on the falling speed. The 3D simulator (`descent_3d.html`) adds Hyderabad temperature and wind, so its landing speed is slightly higher.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

# ---------- inputs (estimates until the CanSat is built and weighed) ----------
m         = 0.30             # kg, total mass
g         = 9.81             # m/s^2
rho       = 1.15             # kg/m^3, air density near 500 m over Hyderabad
Cd_body   = 0.8              # can falling end-first (estimate)
A_body    = np.pi*0.033**2   # m^2, 66 mm can
Cd_chute  = 0.8              # flat canopy (ESA T10)
D_chute   = 0.50             # m
h0        = 500.0            # m, release height
wind      = 3.0              # m/s, steady wind (estimate)
drop_to_detect = 10.0        # m of fall the firmware waits for before firing the burn wire
t_burn    = 2.0              # s, nichrome cuts the thread
t_inflate = 1.0              # s, canopy opens
dt        = 0.01             # s

def fly(m=m, D_chute=D_chute):
    A_chute = np.pi*(D_chute/2)**2
    t, h, v, t_fire, rows = 0.0, h0, 0.0, None, []
    while h > 0:
        if t_fire is None and h < h0 - drop_to_detect:
            t_fire = t
        if t_fire is None or t < t_fire + t_burn:
            A = 0.0
        else:
            A = A_chute * min(1.0, (t - t_fire - t_burn) / t_inflate)
        drag = 0.5*rho*v*abs(v)*(Cd_body*A_body + Cd_chute*A)
        a = g - drag/m                     # + means speeding up downwards
        v += a*dt; h -= v*dt; t += dt
        rows.append((t, h, v, a))
    return pd.DataFrame(rows, columns=["t_s", "alt_m", "speed_ms", "accel_ms2"])

df = fly()
end = df.iloc[-1]
print(f"Time to land:   {end.t_s:.0f} s")
print(f"Landing speed:  {end.speed_ms:.1f} m/s  (rule: 5 to 8 m/s)")
print(f"Top speed:      {df.speed_ms.max():.1f} m/s (before the parachute opens)")
print(f"Opening jerk:   {-df.accel_ms2.min()/g:.0f} g")
print(f"Wind drift:     {wind*end.t_s:.0f} m (assumes it drifts with a steady wind)")
print(f"Telemetry packets (1 per second): {int(end.t_s)}")

## Hand check
Steady falling speed under the parachute: v = √(2mg ÷ (ρ·Cd·A)). It should match the landing speed above.

In [ ]:
A_total = Cd_body*A_body + Cd_chute*np.pi*(D_chute/2)**2
print(f"v = sqrt(2 x {m} x {g} / ({rho} x {A_total:.4f})) = {np.sqrt(2*m*g/(rho*A_total)):.2f} m/s")

## Graphs

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(df.t_s, df.alt_m); ax[0].set(xlabel="time (s)", ylabel="height (m)", title="Height (simulated)")
ax[1].plot(df.t_s, df.speed_ms); ax[1].axhspan(5, 8, alpha=0.2, label="allowed 5-8 m/s")
ax[1].set(xlabel="time (s)", ylabel="falling speed (m/s)", title="Speed (simulated)"); ax[1].legend()
plt.tight_layout(); plt.savefig("descent.png", dpi=150)

## Which parachute size works for which mass

In [ ]:
print("mass(g)  chute(cm)  landing(m/s)  OK?")
for mm in (0.25, 0.30, 0.35):
    for D in (0.40, 0.45, 0.50, 0.55):
        v = fly(mm, D).speed_ms.iloc[-1]
        print(f"{mm*1000:6.0f} {D*100:9.0f} {v:12.1f}   {'yes' if 5 <= v <= 8 else 'no'}")

## Simulated telemetry (1 row per second)

In [ ]:
tele = df.iloc[::int(round(1/dt))].copy()
ground = 505   # m above sea level, Hyderabad (approx.)
tele["pressure_hPa"] = 1013.25*(1 - 2.25577e-5*(tele.alt_m + ground))**5.25588
tele.to_csv("simulated_telemetry.csv", index=False)
tele.head()